<a href="https://colab.research.google.com/github/talhanoor23/generative-ai/blob/main/RAGS/MemoryBased_Rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_openai import ChatOpenAI

/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  exec(code_obj, self.user_global_ns, self.user_ns)


ModuleNotFoundError: No module named 'langchain_openai'

In [ ]:
class RouteQuery(BaseModel):
      datasoure : Literal ["python_docs", "js_docs", "golang_docs"] = Field(
          ...,
          description="Given a user question choose which datasource would be most relevant for answering their question",
      )

In [ ]:
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
structured_llm = llm.with_structured_output(RouteQuery)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
# Prompt
system = """You are an expert at routing a user question to the appropriate data source.

Based on the programming language the question is referring to, route it to the relevant data source."""
prompt=  ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

In [ ]:
router = prompt | structured_llm

In [ ]:
question = """Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""

result = router.invoke({"question": question})

In [ ]:
%pip install -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 2.7 MB/s eta 0:00:00


In [ ]:
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from pydantic.v1 import BaseModel, Field
from langchain_openai import ChatOpenAI

# **Semantic_Routing**

In [ ]:
from langchain.utils.math import cosine_similarity
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Two prompts
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

In [ ]:
embeddings = OpenAIEmbeddings()
prompt_templates = [physics_template, math_template]
embed_templates = embeddings.embed_documents(prompt_templates)

In [ ]:
def prompt_router(input):
  query = embeddings.embed_query(input["query"])
  similarity = cosine_similarity([query], embed_templates)[0]
  # return prompt_templates[similarity.index(max(similarity))]

  most_similar = prompt_templates[similarity.argmax()]
  print(["math_query" if most_similar == math_template else "physics_query" ])

In [ ]:
chain = (
    {"query" : RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
    | StrOutputParser()
)

In [ ]:
quesion = "what is black hole"

In [ ]:
print(chain.invoke({"question" : question}))